In [0]:
#Leyendo la tabla de caso practico ventas
ventas = spark.read.format("delta").table("workspace.electiva.caso_practico_ventas")
# Convertir el DataFrame de Spark a Pandas
ventas = ventas.toPandas()

In [0]:
ventas

In [0]:
## mostrando las primeras 10 filas de la tabla 
ventas.head(10)

In [0]:
# ultimas 5 filas de la tabla 
ventas.tail(5)

In [0]:
# Verificar la cantidad de registros y columnas
print("Número de registros:", ventas.shape[0])
print("Número de columnas:", ventas.shape[1])

In [0]:
ventas.dtypes

nos podemos dar cuenta que las variables de texto estan en formato object y las variables numericas en formato float64

la columna fecha esta en object y tenemos que pasarla al formato adecuado 

In [0]:
#cantidad de valores nulos por columna 
ventas.isnull().sum()

In [0]:
# porcentaje de valores nulos por columna
nulos = ventas.isnull().sum()
porcentaje_nulos = (nulos / len(ventas)) * 100
print(porcentaje_nulos)

In [0]:
ventas[['cantidad', 'precio_unitario']].describe()

In [0]:
# Calculamos las estadísticas descriptivas de cantidad y precio_unitario
estadisticas = ventas[['cantidad', 'precio_unitario']].agg(
    ['mean', 'median', 'std', 'min', 'max'])

estadisticas

**FASE** **2**

In [0]:
# Identificacion de registros duplicados según el id_transaccion
duplicados = ventas['id_transaccion'].duplicated().sum()

print("Número de transacciones duplicadas:", duplicados)

In [0]:
# registros que tienen un id_transaccion repetido
ventas[ventas['id_transaccion'].duplicated(keep=False)].sort_values('id_transaccion')

In [0]:
# Eliminamos registros duplicados según el id_transaccion
ventas = ventas.drop_duplicates(subset=['id_transaccion'])

In [0]:
# Verificacion de que no existen nuevamente id_transaccion duplicados
duplicados = ventas['id_transaccion'].duplicated().sum()

print("Número de transacciones duplicadas después de la limpieza:", duplicados)

In [0]:
# Revisar las categorías existentes en metodo_pago
ventas['metodo_pago'].value_counts()

In [0]:
# se crea una copia independiente de la tabla después de eliminar duplicados
ventas = ventas.drop_duplicates(subset=['id_transaccion']).copy()

In [0]:
# Normalizar la categoría Tarjeta Credito
ventas.loc[:, 'metodo_pago'] = ventas['metodo_pago'].replace({
    'Tarjeta Credito': 'Tarjeta de Crédito'})

In [0]:
# Verificar las categorías de metodo_pago después de la normalización
ventas['metodo_pago'].value_counts()

****Gestion** de **valores** nulos**

In [0]:
# Revisamos nuevamente los valores nulos después de la limpieza
ventas.isnull().sum()

In [0]:
# Calculams la mediana de cantidad para utilizarla en los valores nulos
mediana_cantidad = ventas['cantidad'].median()

print("Mediana de cantidad:", mediana_cantidad)

In [0]:
# Reemplazamos los valores nulos de cantidad por la mediana
ventas['cantidad'] = ventas['cantidad'].fillna(mediana_cantidad)

In [0]:
# Calcular la moda de satisfaccion_cliente para utilizarla en los valores nulos
moda_satisfaccion = ventas['satisfaccion_cliente'].mode()[0]

print("Moda de satisfaccion_cliente:", moda_satisfaccion)

In [0]:
# Reemplazamos los valores nulos de satisfaccion_cliente por la moda
ventas['satisfaccion_cliente'] = ventas['satisfaccion_cliente'].fillna(moda_satisfaccion)

In [0]:
# Verificacion de los valores nulos después de la imputación
ventas.isnull().sum()

Tratamiento de Atípicos

In [0]:
# Se identifica la transacción con el precio_unitario de 9999.00
ventas[ventas['precio_unitario'] == 9999.00]

In [0]:
# Calcular el precio promedio de la categoría Ropa sin incluir el valor atípico
promedio_ropa = ventas[
    (ventas['categoria_producto'] == 'Ropa') &
    (ventas['precio_unitario'] != 9999.00)
]['precio_unitario'].mean()

print("Precio promedio de Ropa:", promedio_ropa)

In [0]:
# Reemplazar el valor atípico de 9999 por el promedio de la categoría Ropa
ventas.loc[
    ventas['precio_unitario'] == 9999.00,
    'precio_unitario'
] = promedio_ropa

In [0]:
# Verificar que el valor atípico haya sido reemplazado
ventas[ventas['precio_unitario'] == 9999.00]

In [0]:
# Se verifica el precio actualizado de la transacción TXN-10060
ventas[ventas['id_transaccion'] == 'TXN-10060']

identificamos un valor atipico de 9999 en una transaccion de la categoria ropa y para conservar el registro, se remplazo este valor por el precio promedio de la categoria ropa, calculado sin incluir el valor atipico

In [0]:
# Calcular el precio promedio por categoría para completar los valores faltantes
# Se excluye el valor atípico de 9999.00 del cálculo
promedio_categoria = ventas[
    (ventas['precio_unitario'].notna()) &
    (ventas['precio_unitario'] != 9999.00)
].groupby('categoria_producto')['precio_unitario'].mean()

print(promedio_categoria)

In [0]:
# Reemplazar los valores nulos de precio_unitario por el promedio de su categoría
ventas['precio_unitario'] = ventas['precio_unitario'].fillna(
    ventas['categoria_producto'].map(promedio_categoria)
)

In [0]:
# Verificar que no queden valores nulos en precio_unitario
ventas['precio_unitario'].isnull().sum()

se completaron los valores faltantes de precio_unitario utilizando el precio promedio de la categoría correspondiente.
De esta forma se conservan los registros y se evitan valores nulos
en los cálculos posteriores.

**FASE 3**

In [0]:
# Calcular el monto total de cada venta
ventas['monto_total'] = ventas['cantidad'] * ventas['precio_unitario']

In [0]:
# se revisan algunos resultados del monto total
ventas[['cantidad', 'precio_unitario', 'monto_total']].head()

conversion

In [0]:
import pandas as pd
# Convertir la fecha a un formato de fecha real
ventas['fecha'] = pd.to_datetime(ventas['fecha'])
# Revisar el tipo de dato de fecha
ventas['fecha'].dtype

In [0]:
# Sacar el mes de la fecha
ventas['mes'] = ventas['fecha'].dt.month

Discretización

In [0]:
# Clasificar las ventas según el monto total
ventas['rango_monto'] = pd.cut(
    ventas['monto_total'],
    bins=[-float('inf'), 100, 500, float('inf')],
    labels=['Bajo', 'Medio', 'Alto'],
    right=False)

In [0]:
# Ver cuántas ventas hay en cada rango
ventas['rango_monto'].value_counts()

**FASE 4**

In [0]:
# Resumir las ventas por categoría de producto
resumen_categoria = ventas.groupby('categoria_producto').agg(
    total_ingresos=('monto_total', 'sum'),
    promedio_unidades=('cantidad', 'mean')
)

resumen_categoria

TABLA DINAMICA

In [0]:
# Crear una tabla con los ingresos por región y categoría
tabla_pivot = pd.pivot_table(
    ventas,
    values='monto_total',
    index='region',
    columns='categoria_producto',
    aggfunc='sum',
    fill_value=0
)

tabla_pivot

distribucion de los ingresos según la región y categoría

los ingresos según la región y categoría

In [0]:
# Calcular la satisfacción promedio según el método de pago
satisfaccion_pago = ventas.groupby('metodo_pago')['satisfaccion_cliente'].mean()

satisfaccion_pago

satisfacción promedio según el método de pago

**FASE 5**

ventas de norte con buena satisfacion

In [0]:
# Filtrar las ventas de la región Norte con satisfacción de 4 o más
ventas_norte = ventas[
    (ventas['region'] == 'Norte') &
    (ventas['satisfaccion_cliente'] >= 4)
]

ventas_norte

In [0]:
# Contar las ventas que cumplen las condiciones
len(ventas_norte)

Muestreo Estratificado

In [0]:
# Tomar una muestra del 10% manteniendo la proporción de cada región
muestra_10 = ventas.groupby('region').sample(frac=0.10, random_state=42)

muestra_10

In [0]:
# Revisar cuántos registros quedaron de cada región
muestra_10['region'].value_counts()

In [0]:
# Revisar el tamaño de la muestra
muestra_10.shape

Se tomó una muestra cercana al 10% de los datos, manteniendo la representación de cada región

YULIAN JAINOVER MONTILLA VELASCO 